In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weights, naanobot_weights,
                                          skeleton_config, naanobot_config, radial_config)

A -- [0.44999999999999996, 0, 0.43, 0.41, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.46, 0.87, 0.71, 1.0]
C -- [0.6, 0, 0.35857142857142854, 0.49, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0.93, 1.37, 0.84, 0.97]
D -- [0.65, -1, 0.21285714285714286, -0.55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.96, 0.54, 1.31, 0.87]
E -- [0.75, -1, 0.23, -0.31, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.46, 0.65, 0.85, 0.89]
F -- [0.8500000000000001, 0, 0.39142857142857146, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1.04, 1.44, 0.7, 0.88]
G -- [0.4, 0, 0.42642857142857143, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.42, 0.36, 1.85, 0.96]
H -- [0.8, 0, 0.5421428571428571, 0.08, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.85, 1.16, 1.02, 0.89]
I -- [0.65, 0, 0.42714285714285716, 1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.86, 1.85, 0.59, 0.76]
K -- [0.75, 1, 0.6764285714285715, -0.23, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.12, 0.86, 0.99, 1.24]
L -- [0.65, 0, 0.42714285714285716, 0.97, 0, 0, 0, 0, 0, 0, 0, 

In [2]:
skeleton_weight_filename = skeleton_weights
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weights
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data()
print(len(pocket_data))
print(type(pocket_data))

3
<class 'dict'>


In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 '3D_skeleton': [[8.160308121825173, -12.641773370947787, 0.12517296474891657],
  [10.739189476216968, -8.490047936799787, -3.758969899105728],
  [8.72477748761848, -9.633089874608249, -4.956009159769427],
  [-3.2058603522778597, -13.064695356811603, 1.1312977666686286],
  [-2.968518162399659, -12.91708738293755, -0.7636469793515633],
  [-12.14781025350629, -3.873186360773401, 1.5466972553638965],
  [-3.2217129444581993, -6.12352515943517, 10.565772474817633],
  [8.42633126329794, -7.412046407639505, 5.249962912202049],
  [1.1374068198154492, -11.955788495780002, 2.6705483574802806],
  [11.353534450003648, 2.2626635012114633, -3.942935780989056],
  [1.2120850740610631, -11.808418591443688, 1.400812432550391],
  [-3.883328488875462, -10.8916478704177, 2.4256257893831603],
  [1.3970575709236328, -9.413788898322721, -6.7197064800611175],
  [9.965323958088957, -4.034736715559789, 3.9174154659895306],
  [1.551245524902129, 11.

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [11]:
import torch
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=10):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = torch.norm(vector - n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:3]}")

        # update context
        aa_id = sorted_distances[0][0]
        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate.tolist() if torch.is_tensor(coordinate) else coordinate]


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_nAAno_features=nb_cfg["n_nAAno_features"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=model._pocket_sph_skeleton())


Position 0:
  raw output: [-0.88  0.28  0.34 -0.05 -1.05  0.13  0.21 -0.36 -0.35 -0.59 -0.16  0.12
  0.2   0.66 -0.19  1.36  0.23  0.32  0.11 -0.6   0.38]
  top 3: [('T', 3.052233934402466), ('A', 3.201112985610962), ('S', 3.203834056854248)]

Position 1:
  raw output: [ 0.49  0.39  0.04 -0.18 -0.07 -0.22  0.08 -0.02 -0.12 -0.18 -0.37  0.01
  0.69  1.22 -0.21  0.83 -0.46 -0.46 -0.71 -0.37  0.6 ]
  top 3: [('S', 3.2073452472686768), ('T', 3.2366065979003906), ('G', 3.2401015758514404)]

Position 2:
  raw output: [-0.08  0.51 -0.4  -0.42 -0.59  0.27  0.08  0.26 -0.36  0.01 -0.11 -0.19
  0.95  1.05 -0.13  1.5  -0.38 -0.79 -0.33 -0.1   0.57]
  top 3: [('S', 3.287566900253296), ('T', 3.356889486312866), ('Y', 3.5008602142333984)]

Position 3:
  raw output: [ 0.35  0.11 -0.54 -0.19 -0.45  0.49  0.32  0.3  -0.54  0.18 -0.29  0.06
  0.43  1.13 -0.07  1.15 -0.46 -0.87 -0.4   0.59 -0.02]
  top 3: [('G', 3.1363370418548584), ('S', 3.234278678894043), ('T', 3.261901378631592)]

Position 4:
  raw 